# Slides em Typst: Pixel e UFABC — versão 2

Este notebook contém os oito templates, imagens e fontes necessários.
Execute as células em ordem. O PDF é produzido a partir de texto estruturado,
sem chamada à API e sem precisar do PowerPoint.

Cada tema oferece **abertura, conceito, comparação e fórmula**.
As configurações ficam separadas da execução. Os arquivos do pacote serão
preparados na pasta escolhida e podem ser editados pelo painel de arquivos.


## 1. Instalação e bibliotecas


In [ ]:
%pip install -q "typst==0.15.0"

import base64
import importlib.util
import io
import json
import zipfile
from pathlib import Path

from google.colab import drive, files
from IPython.display import Image, display


## 2. Parâmetros


In [ ]:
USAR_DRIVE = True
TEMA_ESCOLHIDO = "todos"  # "todos", "pixel" ou "ufabc"
NOME_PDF = "slides_typst_v2.pdf"
GERAR_PREVIAS = True

PASTA_DRIVE = Path("/content/drive/MyDrive/rag_estatistica/templates_typst_v2")
PASTA_LOCAL = Path("/content/rag_estatistica/templates_typst_v2")
PASTA_TEMPLATES = PASTA_DRIVE if USAR_DRIVE else PASTA_LOCAL


## 3. Preparar os templates

A célula abaixo contém o pacote incorporado. Não é necessário editar seu
conteúdo. Ela preserva os arquivos já existentes, inclusive textos que você
tenha alterado no Drive.


In [ ]:
#@title Preparar arquivos, fontes e imagens dos oito templates
if USAR_DRIVE:
    drive.mount("/content/drive")

PASTA_TEMPLATES.mkdir(parents=True, exist_ok=True)
PACOTE_TEMPLATES_B64 = "..."

with zipfile.ZipFile(io.BytesIO(base64.b64decode(PACOTE_TEMPLATES_B64))) as pacote:
    for nome in pacote.namelist():
        caminho = PASTA_TEMPLATES / nome
        caminho.parent.mkdir(parents=True, exist_ok=True)
        if not caminho.exists():
            caminho.write_bytes(pacote.read(nome))

print("Templates disponíveis em:", PASTA_TEMPLATES)


## 4. Conteúdo

Carregue os exemplos. Para experimentar, altere uma string no dicionário
`dados_slides` e execute novamente a geração. Comece mudando apenas o título.

O campo `formula` usa matemática Typst, sem `$`, por exemplo:
`"CV"_(k) = frac(1, k) sum_(i=1)^k "MSE"_i`.
Os outros campos recebem texto simples.


In [ ]:
ARQUIVO_CONTEUDO = PASTA_TEMPLATES / "conteudo.json"

dados_slides = json.loads(ARQUIVO_CONTEUDO.read_text(encoding="utf-8"))
print("Modelos carregados:")
for indice, slide in enumerate(dados_slides["slides"]):
    print(indice, slide["template_id"], list(slide["campos"]))


In [ ]:
# Edite os textos aqui, antes de gerar o PDF.
dados_slides["slides"][0]["campos"]["titulo"] = "Viés e variância na validação cruzada"

# Outro exemplo de alteração:
# dados_slides["slides"][5]["campos"]["conteudo"] = "Sua explicação aqui."


## 5. Gerar o PDF

As posições das caixas permanecem fixas. Os títulos começam em 26 pt e o
corpo em 18 pt, com redução até 24 e 16 pt, respectivamente. Se o texto
ainda não couber, a mensagem identifica o campo: resuma-o ou distribua
o conteúdo em outro slide. Não há quantidade fixa de parágrafos.

Para mudar cores, tamanhos e o limite de redução, edite `estruturas.json`.


In [ ]:
ARQUIVO_PDF = PASTA_TEMPLATES / "output" / NOME_PDF

especificacao = importlib.util.spec_from_file_location(
    "gerador_templates_typst", PASTA_TEMPLATES / "gerar_slides.py"
)
gerador_templates = importlib.util.module_from_spec(especificacao)
especificacao.loader.exec_module(gerador_templates)

resultado_slides = gerador_templates.gerar(
    dados_slides,
    raiz=PASTA_TEMPLATES,
    tema=TEMA_ESCOLHIDO,
    destino=ARQUIVO_PDF,
    previas=GERAR_PREVIAS,
)

print("Slides gerados:", resultado_slides["total_slides"])
print("PDF:", resultado_slides["pdf"])


## 6. Conferir as prévias


In [ ]:
for numero, imagem in enumerate(resultado_slides["previas"], start=1):
    print("SLIDE", numero)
    display(Image(filename=str(imagem), width=960))


## 7. Salvar conteúdo e baixar PDF

Execute esta célula para guardar suas alterações no JSON e baixar o PDF.
Se você trocar o tema ou os textos, gere novamente antes de baixar.


In [ ]:
ARQUIVO_CONTEUDO.write_text(
    json.dumps(dados_slides, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
files.download(str(resultado_slides["pdf"]))
